In [1]:
from ase.io import read
from ase.build import make_supercell
from matplotlib import rc

rc("text", usetex=True)
import matplotlib.pylab as pylab

params = {"legend.fontsize": "14", "axes.labelsize": 20, "axes.titlesize": 20, "xtick.labelsize": 16, "ytick.labelsize": 16}
pylab.rcParams.update(params)
import numpy as np
import glob

try:
    from tqdm import tqdm as _tqdm

    _TQDM_AVAILABLE = True
except ImportError:
    _TQDM_AVAILABLE = False


def _progress(iterable, **kwargs):
    """Wrap an iterable with tqdm if available, else return as-is."""
    if _TQDM_AVAILABLE:
        return _tqdm(iterable, **kwargs)
    return iterable

In [ ]:
# load the molecular dynamics trajectory
traj_directory = "/Users/jackyang-macmini/OneDrive - UNSW/Documents/Projects/artificial_intelligence/LiPSCl-diffusion/langevin_md/FT_MACE_long_traj/600_K_gamma_0_001/"
all_traj_files = glob.glob(traj_directory + "langevin_md_*.traj")
traj = []
for i in range(len(all_traj_files)):
    print(f"Loading trajectory file {i+1}/{len(all_traj_files)}: {all_traj_files[i]}")
    traj_name = traj_directory + f"langevin_md_{i+1}.traj"
    traj += read(traj_name, index=":")
print("loaded", len(traj), "frames")

# Now loop through the trajectory and computed the averaged position of each atom across the trajectory, taking into account the periodic boundary conditions, and make it into a new ASE Atoms object as the reference structure for the essential dynamics analysis.
averaged_structure = traj[0].copy()
for atom in averaged_structure:
    averaged_structure.positions[atom.index] = np.zeros(3)

for frame in _progress(traj, desc="Computing averaged structure", leave=False):
    for atom in frame:
        fractional_position = np.dot(atom.position, np.linalg.inv(frame.cell))
        fractional_position -= np.round(fractional_position)
        wrapped_position = np.dot(fractional_position, frame.cell)
        averaged_structure.positions[atom.index] += wrapped_position
averaged_structure.positions /= len(traj)
print("computed averaged structure")

Loading trajectory file 1/2: /Users/jackyang-macmini/OneDrive - UNSW/Documents/Projects/artificial_intelligence/LiPSCl-diffusion/langevin_md/FT_MACE_long_traj/600_K_gamma_0_001/langevin_md_2.traj
Loading trajectory file 2/2: /Users/jackyang-macmini/OneDrive - UNSW/Documents/Projects/artificial_intelligence/LiPSCl-diffusion/langevin_md/FT_MACE_long_traj/600_K_gamma_0_001/langevin_md_1.traj
loaded 100002 frames


Computing averaged structure:  29%|██▉       | 29419/100002 [03:56<09:31, 123.40it/s]

In [ ]:
# now loop through the trajectory, and for each frame, calculate the displacement of each atom from its position in the equilibrium structure, taking into account the periodic boundary conditions, and normalise the displacement by the square root of the mass of each atom.
atomic_displacements = []
system_energies = []


def compute_atomic_displacements(index, atom):
    ## Calculate displacement from equilibrium position
    ##delta_r = atom.position - equilibrium_structure[index].position

    # calculate displacement from averaged position
    delta_r = atom.position - averaged_structure[index].position
    # Apply periodic boundary conditions
    delta_r -= np.round(delta_r / frame.cell.diagonal()) * frame.cell.diagonal()
    # Normalise by square root of mass
    normalized_displacement = delta_r / np.sqrt(atom.mass)
    return normalized_displacement


for frame in _progress(traj, desc="Compute mass-normalized displacements", leave=False):
    atomic_displacements.append(np.array([compute_atomic_displacements(i, atom) for i, atom in enumerate(frame)]).reshape(-1))
    system_energies.append(frame.get_potential_energy() / frame.get_number_of_atoms())

In [ ]:
covariance_matrix = np.zeros((averaged_structure.get_number_of_atoms() * 3, averaged_structure.get_number_of_atoms() * 3))
atomic_displacements = np.array(atomic_displacements)

for i in _progress(range(averaged_structure.get_number_of_atoms() * 3), desc="Compute covariance matrix", leave=False):
    dispalacement_i = atomic_displacements[:, i]
    covariance_matrix[i, i + 1 :] = np.dot(dispalacement_i, atomic_displacements[:, i + 1 :]) / dispalacement_i.shape[0]

covariance_matrix += covariance_matrix.T + np.identity(averaged_structure.get_number_of_atoms() * 3)

np.save(traj_directory + "covariance_matrix_from_averaged_structure.npy", covariance_matrix)